In [ ]:
# Ячейка 0 — что делать дальше (просто прочитайте вывод после Run)

from IPython.display import Markdown, display

display(
    Markdown(
        """
### PaddleOCR в Colab

1. **Runtime → Change runtime type → GPU** (желательно; на CPU будет дольше).
2. Выполните ячейки **1 → 5** по порядку (кнопка Run). Терминал вручную не нужен.
3. Первый запуск качает модели PaddleOCR — подождите.

**Результаты:** `output/paddle_benchmark/` — тексты `hypotheses/paddle/*.txt`, сводка `paddle_summaries.json` (те же поля метрик, что у MinerU: CER, Final Score, WER в `_diagnostics`).
"""
    )
)
print("Готово: выполняйте ячейку 1.")


In [ ]:
# Ячейка 1 — клон репозитория (Colab) + jiwer

from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path


def in_colab() -> bool:
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


GIT_URL = os.environ.get(
    "OCR_ANALYZE_GIT_URL",
    "https://github.com/developer-mixa/OCR-Analyze.git",
)
REPO_DIR = Path(os.environ.get("OCR_ANALYZE_COLAB_DIR", "/content/OCR-Analyze"))

if in_colab():
    if not (REPO_DIR / "scripts").is_dir():
        print("Клонирую репозиторий…")
        subprocess.check_call(["git", "clone", "--depth", "1", GIT_URL, str(REPO_DIR)])
    os.chdir(REPO_DIR)
    print("Рабочая папка:", Path.cwd().resolve())
else:
    print("Не Colab — откройте ноутбук из корня репозитория. cwd:", Path.cwd().resolve())

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "jiwer"])
print("OK: jiwer. Следующая — ячейка 2 (Paddle + paddleocr).")


In [ ]:
# Ячейка 2 — PaddlePaddle + PaddleOCR (долго, качает веса)

import subprocess
import sys

# True — попытка поставить GPU-сборку (на Colab иногда нужен подбор версии под CUDA; при ошибке поставьте False)
USE_GPU_PADDLE = False

if USE_GPU_PADDLE:
    print("Ставлю paddlepaddle-gpu + paddleocr …")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-U", "paddlepaddle-gpu", "paddleocr"]
    )
else:
    print("Ставлю paddlepaddle (CPU) + paddleocr …")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "paddlepaddle", "paddleocr"])

try:
    import paddle  # noqa: F401

    print("paddle импортируется OK")
except Exception as e:
    print("WARN после установки paddle:", e)
try:
    from paddleocr import PaddleOCR  # noqa: F401

    print("paddleocr импортируется OK")
except Exception as e:
    print("WARN paddleocr:", e)
print("Если импорт не OK — Runtime → Restart session, затем снова ячейки 1–2.")


In [ ]:
# Ячейка 3 — пути, список PNG, заранее создаём каталог результатов

from __future__ import annotations

from pathlib import Path

REPO_ROOT_OVERRIDE: Path | None = None


def find_repo_root() -> Path:
    if REPO_ROOT_OVERRIDE is not None:
        p = REPO_ROOT_OVERRIDE.expanduser().resolve()
        if (p / "scripts").is_dir():
            return p
    cwd = Path.cwd().resolve()
    for start in [cwd, *cwd.parents]:
        if (start / "scripts" / "paddle_image_benchmark.py").is_file():
            return start
    return cwd


REPO_ROOT = find_repo_root()
INPUT_DIR = REPO_ROOT / "input" / "data" / "1"
SCRIPT = REPO_ROOT / "scripts" / "paddle_image_benchmark.py"
OUT_DIR = REPO_ROOT / "output" / "paddle_benchmark"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT.resolve())
print("Скрипт есть:", SCRIPT.is_file())
print("Входные PNG:", INPUT_DIR.resolve(), "— папка есть:", INPUT_DIR.is_dir())
print("Результаты:", OUT_DIR.resolve())

_png = sorted(INPUT_DIR.glob("*.png")) if INPUT_DIR.is_dir() else []
print("Найдено PNG:", len(_png))
for p in _png:
    stem = p.stem
    refs = [
        n
        for n in (f"{stem}.ref.txt", f"{stem}.ref.md", f"{stem}.txt", f"{stem}.md")
        if (INPUT_DIR / n).is_file()
    ]
    print(" ", p.name, "| эталон:", ", ".join(refs) if refs else "нет")
if not _png:
    print("Добавьте PNG в input/data/1.")
print("Следующая — ячейка 4 (язык OCR), затем 5.")


In [ ]:
# Ячейка 4 — язык PaddleOCR (должен совпадать с ячейкой 5)

PADDLE_OCR_LANG = "cyrillic"  # для русского текста; см. доку PaddleOCR: en, ch, …

print("PADDLE_OCR_LANG =", PADDLE_OCR_LANG)
print("При смене языка снова выполните ячейку 5.")


In [ ]:
# Ячейка 5 — прогон PaddleOCR по всем PNG

import json
import subprocess
import sys
from pathlib import Path

if "REPO_ROOT" not in globals() or "SCRIPT" not in globals():
    raise RuntimeError("Сначала ячейка 3.")

lang = globals().get("PADDLE_OCR_LANG", "cyrillic")

argv = [
    sys.executable,
    str(SCRIPT),
    "--input-dir",
    str(INPUT_DIR),
    "--output-dir",
    str(OUT_DIR),
    "--lang",
    str(lang),
]

print("Запуск PaddleOCR, язык:", lang)
print("Пишем в", OUT_DIR)
subprocess.check_call(argv, cwd=str(REPO_ROOT))

hyp_dir = OUT_DIR / "hypotheses" / "paddle"
print("\nГотовые тексты (.txt):")
txts = sorted(hyp_dir.glob("*.txt"))
for p in txts:
    print(" ", p.name, p.stat().st_size, "байт")
if not txts:
    print("  нет .txt — см. paddle_runs.jsonl (поле error)")

for name in ("paddle_hypotheses_raw.json", "paddle_hypotheses_concat.txt"):
    fp = OUT_DIR / name
    if fp.is_file():
        print("Сводка сырого текста:", fp.name, fp.stat().st_size, "байт")

summ = OUT_DIR / "paddle_summaries.json"
if summ.is_file():
    print("\n---", summ.name, "---")
    txt = summ.read_text(encoding="utf-8")
    print(txt)
    try:
        outs = json.loads(txt).get("paddleocr", {}).get("_outputs")
        if outs:
            print("\nПути (_outputs):")
            for k, v in outs.items():
                print(" ", k, "→", v)
    except json.JSONDecodeError:
        pass

print("\nГотово.")
